# HDBSCAN clustering

Clusters all feature titles using the shared similarity matrix from `3_similarity.ipynb`. This follows the same workflow as `6_dbscan.ipynb`, but HDBSCAN can find clusters with different densities and marks outliers as noise.

**Method:** convert similarity to distance, compare a small grid of HDBSCAN parameters, inspect the selected clusters, evaluate silhouette on non-noise points, and visualize with t-SNE.

**Requirement:** HDBSCAN is included in scikit-learn 1.3 and later.

In [1]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

MIN_CLUSTER_SIZE_VALUES = [3, 5, 8, 12, 20]
MIN_SAMPLES_VALUES = [2, 3, 5, 8]
MIN_COVERAGE = 0.60
SEED = 0
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = np.clip(1 - similarity, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} attributes, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


## 1. Parameter search

`min_cluster_size` sets the smallest group HDBSCAN may return. `min_samples` controls how conservatively it treats points as noise; larger values usually label more points as noise.

We require at least 60% coverage, meaning at least 60% of the items must be assigned to clusters instead of noise. Among runs meeting this minimum and producing at least two clusters, we rank by silhouette score (higher is better), then by noise ratio (lower is better).

In [2]:
def evaluate_hdbscan(labels, distances):
    non_noise_mask = labels != -1
    n_noise = int((~non_noise_mask).sum())
    noise_ratio = n_noise / len(labels)
    coverage = float(non_noise_mask.mean())

    non_noise_labels = labels[non_noise_mask]
    n_clusters = len(set(non_noise_labels.tolist()))
    if n_clusters < 2:
        return None

    reduced_distances = distances[np.ix_(non_noise_mask, non_noise_mask)]
    silhouette = silhouette_score(
        reduced_distances,
        non_noise_labels,
        metric="precomputed",
    )

    return {
        "silhouette": float(silhouette),
        "noise_ratio": float(noise_ratio),
        "coverage": coverage,
        "n_clusters": int(n_clusters),
        "n_noise": int(n_noise),
    }


candidates = []
for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        if min_samples > min_cluster_size:
            continue

        model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="precomputed",
            copy=True,
        )
        labels = model.fit_predict(distance_matrix)
        evaluation = evaluate_hdbscan(labels, distance_matrix)
        if evaluation is None or evaluation["coverage"] < MIN_COVERAGE:
            continue

        candidates.append(
            {
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                **evaluation,
            }
        )

if not candidates:
    raise ValueError(f"No HDBSCAN configuration produced at least 2 clusters with {MIN_COVERAGE:.0%} coverage.")

candidates = sorted(
    candidates,
    key=lambda row: (-row["silhouette"], row["noise_ratio"], row["n_clusters"]),
)
best = candidates[0]

print("Top 10 parameter sets (sorted by silhouette desc, noise asc):")
for row in candidates[:10]:
    print(
        f"min_cluster_size={row['min_cluster_size']}, min_samples={row['min_samples']} | "
        f"silhouette={row['silhouette']:.4f}, clusters={row['n_clusters']}, "
        f"coverage={100 * row['coverage']:.1f}%, "
        f"noise={row['n_noise']} ({100 * row['noise_ratio']:.1f}%)"
    )

print("\nSelected configuration:")
print(
    f"min_cluster_size={best['min_cluster_size']}, min_samples={best['min_samples']} | "
    f"silhouette={best['silhouette']:.4f}, clusters={best['n_clusters']}, "
    f"coverage={100 * best['coverage']:.1f}%, "
    f"noise={best['n_noise']} ({100 * best['noise_ratio']:.1f}%)"
)

Top 10 parameter sets (sorted by silhouette desc, noise asc):
min_cluster_size=3, min_samples=3 | silhouette=0.2963, clusters=38, noise=236 (53.0%)
min_cluster_size=3, min_samples=2 | silhouette=0.2360, clusters=61, noise=147 (33.0%)
min_cluster_size=5, min_samples=2 | silhouette=0.2249, clusters=30, noise=192 (43.1%)
min_cluster_size=5, min_samples=3 | silhouette=0.2190, clusters=20, noise=223 (50.1%)
min_cluster_size=8, min_samples=2 | silhouette=0.1657, clusters=14, noise=218 (49.0%)
min_cluster_size=8, min_samples=8 | silhouette=0.1531, clusters=3, noise=310 (69.7%)
min_cluster_size=12, min_samples=8 | silhouette=0.1531, clusters=3, noise=310 (69.7%)
min_cluster_size=8, min_samples=5 | silhouette=0.1373, clusters=5, noise=250 (56.2%)
min_cluster_size=12, min_samples=5 | silhouette=0.1171, clusters=3, noise=269 (60.4%)
min_cluster_size=20, min_samples=5 | silhouette=0.1149, clusters=2, noise=283 (63.6%)

Selected configuration:
min_cluster_size=3, min_samples=3 | silhouette=0.2963, 

## 2. Fit HDBSCAN and inspect groups

In [3]:
final_model = HDBSCAN(
    min_cluster_size=best["min_cluster_size"],
    min_samples=best["min_samples"],
    metric="precomputed",
    copy=True,
)
labels = final_model.fit_predict(distance_matrix)

sizes = Counter(labels.tolist())
cluster_labels = [label for label in sorted(sizes) if label != -1]
n_noise = sizes.get(-1, 0)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"coverage: {100 * (len(labels) - n_noise) / len(labels):.1f}% | "
    f"noise points: {n_noise} / {len(labels)} ({100 * n_noise / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[sizes[label] for label in sorted(cluster_labels, key=lambda label: -sizes[label])]}")

members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    print(f"\nCluster {label} ({len(members)} attributes)")
    for member in members:
        print(f"  - {member}")

Clusters (excluding noise): 38 | noise points: 236 / 445 (53.0%)
Cluster sizes (largest first): [20, 12, 10, 9, 9, 9, 8, 7, 7, 7, 7, 6, 6, 6, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]

Cluster 34 (20 attributes)
  - Light insulation
  - Windproof Front Panel
  - Insulating
  - Insulating fleece inner
  - Windproof protection
  - Deep winter insulation
  - Thermal fleece fabric
  - Deep winter protection
  - Cool weather insulation
  - Windproof softshell 3-layer panels
  - Windtex Softshell 3-layer fabric
  - Windproof front panel
  - Windproof softshell fabric
  - Winter insulation
  - Highly insulating
  - Thermal insulation
  - Windproof front panels
  - Deep winter windproof protection
  - Premium deep winter insulation
  - Windproof

Cluster 7 (12 attributes)
  - Practical leg pocket
  - Convenient internal pocket
  - Convenient rear pockets
  - Rear pockets
  - Practical pockets
  - Three rear pockets plus zipped pocket
  - Zipped leg pocket
  - Deep

## 3. Clustering evaluation

As with DBSCAN, noise points (`-1`) are excluded from the silhouette score. Coverage is the share of items assigned to clusters; the parameter search requires at least 60%.

In [4]:
non_noise_mask = labels != -1
non_noise_labels = labels[non_noise_mask]
coverage = float(non_noise_mask.mean())

print(f"Non-noise points: {int(non_noise_mask.sum())}")
print(f"Noise points: {int((~non_noise_mask).sum())}")
print(f"Coverage: {100 * coverage:.1f}% (minimum: {100 * MIN_COVERAGE:.0f}%)")
print(f"Clusters (excluding noise): {len(set(non_noise_labels.tolist()))}")

if len(set(non_noise_labels.tolist())) >= 2:
    reduced_distances = distance_matrix[np.ix_(non_noise_mask, non_noise_mask)]
    silhouette = silhouette_score(reduced_distances, non_noise_labels, metric="precomputed")
    print(f"Silhouette score (non-noise only): {silhouette:.4f}")
else:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")

Non-noise points: 209
Noise points: 236
Clusters (excluding noise): 38
Silhouette score (non-noise only): 0.2963


## 4. Cluster visualization

t-SNE projects the precomputed distances (`1 - similarity`) to two dimensions for display only. The largest clusters get highlight colors, smaller clusters are grey, and noise points are shown as hollow markers.

In [6]:
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    learning_rate="auto",
    random_state=SEED,
).fit_transform(distance_matrix)

PALETTE = ["#2f6b8a", "#e07a5f", "#3d9970", "#8e6cbb", "#d4a72c", "#d45d79", "#5086c1", "#8c6d5c"]
ranked_clusters = sorted(cluster_labels, key=lambda label: (-sizes[label], label))
highlighted_clusters = ranked_clusters[:HIGHLIGHT_GROUPS]
group_names = {label: f"Cluster {label}" for label in cluster_labels}

def add_points(indices, name, color, hollow=False, size=8):
    if len(indices) == 0:
        return

    marker = {"size": size, "color": color}
    if hollow:
        marker.update({"symbol": "circle-open", "line": {"color": color, "width": 1.5}})

    figure.add_trace(
        go.Scatter(
            x=coordinates[indices, 0],
            y=coordinates[indices, 1],
            mode="markers",
            name=name,
            marker=marker,
            text=[attributes[index] for index in indices],
            customdata=["Noise" if labels[index] == -1 else group_names[labels[index]] for index in indices],
            hovertemplate="<b>%{text}</b><br>%{customdata}<extra></extra>",
        )
    )

figure = go.Figure()
noise_indices = np.where(labels == -1)[0]
add_points(noise_indices, f"Noise ({len(noise_indices)})", "#b8bec5", hollow=True, size=7)

other_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and label not in highlighted_clusters],
    dtype=int,
)
add_points(other_indices, "Other clusters", "#b8bec5", size=7)

for color, label in zip(PALETTE, highlighted_clusters):
    indices = np.where(labels == label)[0]
    add_points(indices, f"Cluster {label} ({sizes[label]})", color, size=10)

figure.update_layout(
    title=dict(text="Attribute clusters: HDBSCAN (t-SNE)", x=0),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", itemsizing="constant"),
    hoverlabel=dict(bgcolor="white"),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))